In [1]:
from   osgeo     import gdal
from   shapely   import points
import geopandas as gpd
import numpy as np
import time
import os

In [7]:
# Input Directories
Input_Rasters_Directory = 'C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Mapping/0_ALS/Spain/'
Output_GPKG_Direcory    = 'C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Mapping/'

In [9]:
# Get data files to process
Rasters_Paths_To_Process = [os.path.join(Input_Rasters_Directory, Raster_Path)
                            for Raster_Path in os.listdir(Input_Rasters_Directory)
                            if Raster_Path.endswith('.tif')]

In [11]:
# Count time
Time_Process_Start = time.perf_counter()

# Process all rasters
for Raster_Path in Rasters_Paths_To_Process:
    # Count iteration time
    Time_Iteration_Start = time.perf_counter()

    print(Raster_Path.split('/')[-1])
    # ----------------------------
    # 1 - Get Raster File Data - 1
    print('Obtaining raster data...')
    Raster              = gdal.Open(Raster_Path)
    Raster_CRS          = Raster.GetProjection()
    Raster_GeoTransform = Raster.GetGeoTransform()
    Raster_Band         = Raster.GetRasterBand(1)
    Raster_Data         = Raster_Band.ReadAsArray()
    Raster_NoData_Value = Raster_Band.GetNoDataValue()
    # ----------------------------
    # 2 - Get pixels positions - 2
    print('Getting raster geographic positions...')
    # Get Rows & Columns indices
    Columns_Indices, Rows_Indices = np.meshgrid(np.arange(Raster.RasterXSize), np.arange(Raster.RasterYSize), indexing='xy')
    # Convert pixel indices to real-world coordinates
    X_Coordinates = Raster_GeoTransform[0] + Columns_Indices * Raster_GeoTransform[1] + Rows_Indices * Raster_GeoTransform[2]
    Y_Coordinates = Raster_GeoTransform[3] + Columns_Indices * Raster_GeoTransform[4] + Rows_Indices * Raster_GeoTransform[5]
    # Calculate Pixels centroid coordinates
    X_Coordinates = X_Coordinates + Raster_GeoTransform[1] / 2
    Y_Coordinates = Y_Coordinates + Raster_GeoTransform[5] / 2
    # Close Raster not required anymore
    Raster = None
    # ----------------------------
    # 3 - Create Geopandas Dataframe with Data - 3
    print('Creating geopandas dataframe...')
    # Filter out No Data values & Flatten array (assume that are all the negative values, for one band the no-data are the higher values)
    X_Coordinates_Flatten = X_Coordinates[(Raster_Data > 0) & (Raster_Data < 30_000)]
    Y_Coordinates_Flatten = Y_Coordinates[(Raster_Data > 0) & (Raster_Data < 30_000)]
    Raster_Data_Flatten   = Raster_Data[(Raster_Data > 0) & (Raster_Data < 30_000)]
    # Create Geometry to store in Geopandas Dataframe (using Shapely)
    GPKG_Geometry = points(X_Coordinates_Flatten, Y_Coordinates_Flatten)
    # Build GeoDataFrame
    BIOMASS_POOL   = Raster_Path.split('_')[-2]
    GPKG_Dataframe = gpd.GeoDataFrame({BIOMASS_POOL: Raster_Data_Flatten}, geometry=GPKG_Geometry, crs=Raster_CRS)
    # ----------------------------
    # 4 - Save Geopandas Dataframe to Geopackage - 4
    print('Saving geopandas dataframe to geopackage...')
    Output_Path = Output_GPKG_Direcory + Raster_Path.split('/')[-1].replace('.tif', '.gpkg')
    GPKG_Dataframe.to_file(Output_Path, driver="GPKG")
    print('Process concluded.')

    # Compute iteration processing time
    Time_Iteration_Ends = time.perf_counter()
    Processing_Time     = Time_Iteration_Ends - Time_Iteration_Start
    # Convert to h:m:s
    hours, rem          = divmod(Processing_Time, 3600)
    minutes, seconds    = divmod(rem, 60)
    print(f"Elapsed iteration time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")
    print('+-+-+-+-+-+')

# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

Extremadura_20180000_ALS_AGBD_ALL.tif
Obtaining raster data...
Getting raster geographic positions...
Creating geopandas dataframe...
Saving geopandas dataframe to geopackage...


ValueError: Could not correctly detect GDAL data files installed by pyogrio wheel

In [ ]:
Raster_NoData_Value